#warehouse setup

In [0]:
%sql
CREATE CATALOG data_warehouse;
CREATE SCHEMA data_warehouse.bndes;
CREATE VOLUME data_warehouse.bndes.file_injest;

#job config

In [0]:
%pip install --upgrade databricks-sdk==0.70.0
%restart_python

from databricks.sdk.service.jobs import JobSettings as Job


DATA_MASTER_PIPELINE = Job.from_dict(
    {
        "name": "DATA_MASTER_PIPELINE_2",
        "trigger": {
            "pause_status": "UNPAUSED",
            "file_arrival": {
                "url": "/Volumes/data_warehouse/bndes/files_ingest/",
                "min_time_between_triggers_seconds": 30,
                "wait_after_last_change_seconds": 30,
            },
        },
        "tasks": [
            {
                "task_key": "00_FILE_INGEST",
                "notebook_task": {
                    "notebook_path": "00.file_ingest",
                    "source": "GIT",
                },
            },
            {
                "task_key": "01_BRONZE_TABLE",
                "depends_on": [
                    {
                        "task_key": "00_FILE_INGEST",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "01.bronze_table",
                    "source": "GIT",
                },
                "email_notifications": {
                    "on_failure": [
                        "fernandomiguel99@gmail.com",
                    ],
                },
            },
            {
                "task_key": "02_SILVER_TABLE",
                "depends_on": [
                    {
                        "task_key": "01_BRONZE_TABLE",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "02.silver_table",
                    "source": "GIT",
                },
                "email_notifications": {
                    "on_failure": [
                        "fernandomiguel99@gmail.com",
                    ],
                },
            },
            {
                "task_key": "03_GOLD_TABLE",
                "depends_on": [
                    {
                        "task_key": "02_SILVER_TABLE",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "03.gold_table",
                    "source": "GIT",
                },
                "email_notifications": {
                    "on_failure": [
                        "fernandomiguel99@gmail.com",
                    ],
                },
            },
        ],
        "git_source": {
            "git_url": "https://github.com/fernandomiguel99/case_data_master_data_eng.git",
            "git_provider": "gitHub",
            "git_branch": "main",
        },
        "queue": {
            "enabled": True,
        },
        "environments": [
            {
                "environment_key": "PIPELINE_ENV",
                "spec": {
                    "environment_version": "5",
                },
            },
        ],
        "performance_target": "PERFORMANCE_OPTIMIZED",
    }
)

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
try:
    w.jobs.reset(new_settings=DATA_MASTER_PIPELINE, job_id=87138494386020)
except:
    w.jobs.create(**DATA_MASTER_PIPELINE.as_shallow_dict())
